In [ ]:
%sql
DROP TABLE IF EXISTS workspace.gold_weather.kpi_climate_risk_daily;

CREATE TABLE IF NOT EXISTS workspace.gold_weather.kpi_climate_risk_daily (
  location_id BIGINT,
  date_key BIGINT,
  data_type STRING,
  frost_alert BOOLEAN,
  heat_alert BOOLEAN,
  heavy_rain_alert BOOLEAN
)

In [ ]:
# Umbral unico de calor (30 C) para todo el pais es el supuesto mas debil del
# modelo (Puno vs Piura no son comparables), ver Proyecto/DECISIONS.md #6
kpi = spark.sql("""
    SELECT
        location_id,
        date_key,
        data_type,
        temp_min <= 0 AS frost_alert,
        temp_max >= 30 AS heat_alert,
        precipitation_sum >= 20 AS heavy_rain_alert
    FROM workspace.gold_weather.fact_weather_daily
""")

kpi.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_weather.kpi_climate_risk_daily")

In [ ]:
display(kpi.limit(5))